In [ ]:
import pandas as pd
import numpy as np
import os
from itertools import combinations

In [ ]:
def get_link_list(dir):
    linklist = []
    for file in os.listdir(dir):
        link = file.split(' ')[4]
        linklist.append(link)
    return linklist

In [ ]:
def outlier_removal(df, column):

    df[column] = df[column].replace(-1, np.nan)

    r = df[column].dropna().to_numpy()
    
    if r.size == 0:
        print("Coluna não contém valores suficientes para análise.")
        return df

    r_max = np.max(r) 
    r = r / r_max  

    perc_min = []
    p_min = np.linspace(0.1, 2, 20)
    for i in p_min:
        perc_min.append(np.percentile(r, i))
    diff_perc_min = np.diff(perc_min)
    index_min = np.argmax(diff_perc_min)  
    thres_min = np.mean(perc_min[index_min:index_min + 2])

    perc_max = []
    p_max = np.linspace(98, 100, 20)
    for i in p_max:
        perc_max.append(np.percentile(r, i))
    diff_perc_max = np.diff(perc_max)
    index_max = np.argmax(diff_perc_max)  
    thres_max = np.mean(perc_max[index_max:index_max + 2])

    r_filtered = np.where((r < thres_min) | (r > thres_max), np.nan, r)

    r_filtered = r_filtered * r_max  

    df_filtered = df.copy()
    df_filtered.loc[~df[column].isna(), column] = r_filtered

    return df_filtered


In [ ]:
#comparar se os datasets tem o mesmo valor, fora os valores imputados 
caminho_pasta_original = '../datasets/treated longest interval with failures'
caminho_pasta_imputado = '../datasets/imputed-treated-longest-interval-with-failures'

tecnicas = ['interpolacao-linear', 'media-movel','mediana-movel', 'knn', 'svd']

lista_tcp = ['bbr']  
# lista_link = ['ap-ma', 'ba-es', 'es-ba', 'rj-ap', 'es-ba', 'es-ce', 'rj-es', 'rj-sc']  

lista_link = get_link_list(caminho_pasta_original)

def encontrar_arquivo(pasta, substrings):
    arquivos_encontrados = []
    for arquivo in os.listdir(pasta):
        if arquivo.endswith('.csv') and all(substring in arquivo for substring in substrings):
            arquivos_encontrados.append(os.path.join(pasta, arquivo))
    return arquivos_encontrados


def comparar_vazao(caminho_pasta_original, caminho_pasta_imputado, lista_tcp, lista_link, tecnicas):
    import numpy as np
    import pandas as pd
    
    for tcp in lista_tcp:
        for tecnica in tecnicas:
            caminho_pasta_imputado_tecnica = os.path.join(caminho_pasta_imputado, tecnica)
            for link in lista_link:
                arquivo_original = encontrar_arquivo(caminho_pasta_original, [tcp, link])
                arquivo_imputado = encontrar_arquivo(caminho_pasta_imputado_tecnica, [tcp, link])
                
                if arquivo_original and arquivo_imputado:
                    arquivo_original = arquivo_original[0]
                    arquivo_imputado = arquivo_imputado[0]
                    
                    df_original = pd.read_csv(arquivo_original)
                    df_original = outlier_removal(df_original, 'Throughput')
                    df_imputado = pd.read_csv(arquivo_imputado)
    
                    if 'Throughput' not in df_original.columns or 'Throughput' not in df_imputado.columns:
                        print(f"Coluna 'Throughput' não encontrada nos arquivos para tcp={tcp} e link={link}.")
                        continue

                    # Convert 'Timestamp' columns to datetime
                    df_original['Timestamp'] = pd.to_datetime(df_original['Timestamp'])
                    df_imputado['Timestamp'] = pd.to_datetime(df_imputado['Timestamp'])

                    # Perform an outer merge to check for mismatches
                    merged_df = df_original[['Timestamp', 'Throughput']].merge(
                        df_imputado[['Timestamp', 'Throughput']],
                        on='Timestamp',
                        how='outer',
                        suffixes=('_original', '_imputado'),
                        indicator=True
                    )

                    # Check for mismatches
                    if merged_df['_merge'].value_counts().get('both', 0) == 0:
                        print(f"Nenhum 'Timestamp' comum encontrado para tcp={tcp}, link={link}, técnica={tecnica}.")
                        continue

                    # Proceed with only the common timestamps
                    merged_df = merged_df[merged_df['_merge'] == 'both']

                    # Create mask for non-null original 'Throughput' values
                    mask_nao_nulo = ~merged_df['Throughput_original'].isna()

                    # Select non-null 'Throughput' values
                    throughput_original = merged_df.loc[mask_nao_nulo, 'Throughput_original']
                    throughput_imputado = merged_df.loc[mask_nao_nulo, 'Throughput_imputado']

                    # Ensure indices are aligned
                    throughput_imputado = throughput_imputado[throughput_original.index]

                    # Compare using numpy.isclose with equal_nan=True
                    throughput_igual = np.isclose(throughput_original, throughput_imputado, equal_nan=True).all()

                    if throughput_igual:
                        print(f"Os dados da coluna 'Throughput' para tcp={tcp}, link={link}, técnica={tecnica} são idênticos nas linhas não nulas.")
                    else:
                        diferencas = ~np.isclose(throughput_original, throughput_imputado, equal_nan=True)
                        df_diferencas = pd.DataFrame({
                            'Timestamp': merged_df.loc[mask_nao_nulo, 'Timestamp'][diferencas],
                            'throughput_original': throughput_original[diferencas].values,
                            'throughput_imputado': throughput_imputado[diferencas].values
                        })
                        print(f"Diferenças encontradas para tcp={tcp}, link={link}, técnica={tecnica}:")
                        print(df_diferencas)
                else:
                    print(f"Arquivos para tcp={tcp}, link={link}, técnica={tecnica} não encontrados.")



comparar_vazao(caminho_pasta_original, caminho_pasta_imputado, lista_tcp, lista_link, tecnicas)


In [ ]:
len(lista_link)

In [ ]:
#verificando se as imputações de diferentes metodos estao iguais
# caminho_pasta_original = '../datasets/melhores-tratados'
# caminho_pasta_imputado_base = '../datasets/dados-vazao-imputados' 

# lista_tcp = ['bbr', 'cubic']  
# lista_link = ['ap-ma', 'ba-es', 'es-ba', 'rj-ap', 'es-ba', 'es-ce', 'rj-es', 'rj-sc']  

def encontrar_arquivo(pasta, substrings):
    arquivos_encontrados = []
    for arquivo in os.listdir(pasta):
        if arquivo.endswith('.csv') and all(substring in arquivo for substring in substrings):
            arquivos_encontrados.append(os.path.join(pasta, arquivo))
    return arquivos_encontrados


def comparar_imputacao_par_a_par(caminho_pasta_original, caminho_pasta_imputado_base, lista_tcp, lista_link):
    for tcp in lista_tcp:
        for link in lista_link:
            arquivo_original = encontrar_arquivo(caminho_pasta_original, [tcp, link])
            
            if arquivo_original:
                arquivo_original = arquivo_original[0]
                
                # Ler o CSV sem 'date_parser'
                df_original = pd.read_csv(arquivo_original)
                
                # Converter 'Timestamp' para datetime
                df_original['Timestamp'] = pd.to_datetime(df_original['Timestamp'], format='%d-%m-%y %H:%M:%S')
                
                # Aplicar remoção de outliers
                df_original = outlier_removal(df_original, 'Throughput')
                
                # Identificar timestamps onde 'Throughput' é NaN após a remoção de outliers
                timestamps_nan = df_original[df_original['Throughput'].isna()]['Timestamp']
                
                # Inicializar DataFrame para coletar imputações
                df_imputacoes = pd.DataFrame({'Timestamp': timestamps_nan}).set_index('Timestamp')
                
                for metodo in os.listdir(caminho_pasta_imputado_base):
                    caminho_metodo = os.path.join(caminho_pasta_imputado_base, metodo)
                    if os.path.isdir(caminho_metodo): 
                        arquivo_imputado = encontrar_arquivo(caminho_metodo, [tcp, link])
                        
                        if arquivo_imputado:
                            arquivo_imputado = arquivo_imputado[0]
                            
                            # Ler o CSV sem 'date_parser'
                            df_imputado = pd.read_csv(arquivo_imputado)
                            
                            # Converter 'Timestamp' para datetime
                            df_imputado['Timestamp'] = pd.to_datetime(df_imputado['Timestamp'], format='%d-%m-%y %H:%M:%S')
                            
                            # Ajustar índice para 'Timestamp' para alinhamento
                            df_imputado = df_imputado[['Timestamp', 'Throughput']].set_index('Timestamp')
                            
                            # Obter imputações nos timestamps onde os dados originais são NaN usando reindex
                            imputacoes_metodo = df_imputado.reindex(timestamps_nan)
                            
                            # Renomear coluna 'Throughput' para o nome do método
                            imputacoes_metodo = imputacoes_metodo.rename(columns={'Throughput': metodo})
                            
                            # Unir imputações no df_imputacoes
                            df_imputacoes = df_imputacoes.join(imputacoes_metodo, how='left')
                
                # Agora df_imputacoes possui colunas para cada método, indexado por Timestamp
                
                # Comparar imputações par a par
                for metodo1, metodo2 in combinations(df_imputacoes.columns, 2):
                    # Obter imputações para ambos os métodos
                    imputacao1 = df_imputacoes[metodo1]
                    imputacao2 = df_imputacoes[metodo2]
                    
                    # Selecionar índices onde ambos os métodos têm imputações não NaN
                    mask_non_nan = imputacao1.notna() & imputacao2.notna()
                    
                    if mask_non_nan.any():
                        # Comparar imputações nesses índices
                        iguais = np.isclose(imputacao1[mask_non_nan], imputacao2[mask_non_nan], equal_nan=True).all()
                        
                        if iguais:
                            print(f"Valores imputados são iguais entre {metodo1} e {metodo2} para tcp={tcp} e link={link}.")
                            # Opcionalmente, imprimir o DataFrame
                            print(df_imputacoes[[metodo1, metodo2]].loc[mask_non_nan])
                        else:
                            print(f"Valores diferentes entre {metodo1} e {metodo2} para tcp={tcp} e link={link}.")
                    else:
                        print(f"Não há valores comparáveis entre {metodo1} e {metodo2} para tcp={tcp} e link={link}.")
            else:
                print(f"Arquivo original para tcp={tcp} e link={link} não encontrado.")


comparar_imputacao_par_a_par(caminho_pasta_original, caminho_pasta_imputado, lista_tcp, lista_link)
